In [1]:
!pip install torch scikit-learn matplotlib

In [2]:

# ============================================================
# FFD-STRA-GAT: Financial Fraud Detection
# Spatiotemporal Risk-Aware Graph Attention Networks
# Single-cell implementation | PyTorch + DDP
# ============================================================

import os, math, time, copy, random, warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = "./outputs"          # change to "/kaggle/working" on Kaggle
os.makedirs(OUTPUT_DIR, exist_ok=True)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    confusion_matrix, balanced_accuracy_score,
    roc_curve, precision_recall_curve,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# HYPERPARAMETERS
# ============================================================
CFG = {
    "n_accounts":    800,
    "n_transactions": 6000,
    "fraud_ratio":   0.15,
    "hidden_dim":    64,
    "num_layers":    3,
    "dropout":       0.30,
    "type_emb_dim":  16,
    "epochs":        250,
    "lr":            3e-4,
    "weight_decay":  1e-4,
    "patience":      40,        # longer patience — stops FFD early-stopping too soon
    "train_ratio":   0.60,      # harder split: 60/20/20 train/val/test
    "val_ratio":     0.20,
    "use_ddp":       True,
    "world_size":    2,
}

# ============================================================
# ACCOUNT TYPES (30)
# ============================================================
ACCOUNT_TYPES = [
    "Individual", "Joint", "Minor", "Senior_Citizen", "Salary", "Pension",
    "NRE", "NRO", "FCNR",
    "Proprietorship", "Partnership", "LLP", "OPC",
    "Private_Ltd", "Public_Ltd", "Foreign_Company", "HUF",
    "Trust", "Society", "Club", "NGO",
    "Educational_Institution", "Religious_Institution",
    "Government_Dept", "Municipality", "Local_Body",
    "Cooperative_Society", "Cooperative_Bank",
    "Escrow", "Merchant",
]
NUM_TYPES = len(ACCOUNT_TYPES)
assert NUM_TYPES == 30

STATIC_THRESHOLDS = {
    "Individual": 0.55,             "Joint": 0.58,
    "Minor": 0.45,                  "Senior_Citizen": 0.48,
    "Salary": 0.60,                 "Pension": 0.50,
    "NRE": 0.42,                    "NRO": 0.40,
    "FCNR": 0.38,                   "Proprietorship": 0.52,
    "Partnership": 0.50,            "LLP": 0.53,
    "OPC": 0.55,                    "Private_Ltd": 0.48,
    "Public_Ltd": 0.50,             "Foreign_Company": 0.35,
    "HUF": 0.55,                    "Trust": 0.45,
    "Society": 0.50,                "Club": 0.52,
    "NGO": 0.48,                    "Educational_Institution": 0.55,
    "Religious_Institution": 0.50,  "Government_Dept": 0.65,
    "Municipality": 0.63,           "Local_Body": 0.62,
    "Cooperative_Society": 0.53,    "Cooperative_Bank": 0.55,
    "Escrow": 0.40,                 "Merchant": 0.42,
}

# ============================================================
# W0 : 30×30 ASYMMETRIC INTERACTION WEIGHT MATRIX
# ============================================================
def build_W0(seed=42):
    rng = np.random.default_rng(seed)
    W0  = rng.beta(2, 5, (NUM_TYPES, NUM_TYPES)).astype(np.float32)
    for i in range(NUM_TYPES):
        W0[i, i] = float(rng.beta(5, 2))
    noise = rng.uniform(-0.08, 0.08, (NUM_TYPES, NUM_TYPES)).astype(np.float32)
    np.fill_diagonal(noise, 0.0)
    return torch.tensor(np.clip(W0 + noise, 0.01, 0.99), dtype=torch.float32)

W0 = build_W0()

# ============================================================
# SYNTHETIC TRANSACTION GRAPH
# ============================================================
def generate_transaction_graph(n_accounts=800, n_transactions=6000,
                                fraud_ratio=0.15, seed=0,
                                device=torch.device("cpu")):
    rng = np.random.default_rng(seed)

    # Node attributes
    cat_idx = rng.integers(0, NUM_TYPES, n_accounts).astype(np.int64)
    H       = rng.uniform(0.1, 1.0,  n_accounts).astype(np.float32)
    age     = (rng.uniform(18, 80,   n_accounts) / 80).astype(np.float32)
    loc     = (rng.integers(0, 50,   n_accounts) / 50).astype(np.float32)
    T_avg_n = rng.uniform(1000, 1e5, n_accounts).astype(np.float32)
    eps_n   = rng.uniform(0.10, 0.60, n_accounts).astype(np.float32)

    # Pre-assign fraud accounts (exactly fraud_ratio fraction)
    n_fraud_accs   = int(n_accounts * fraud_ratio)
    fraud_accs     = set(rng.choice(n_accounts, n_fraud_accs, replace=False).tolist())

    # Generate edges
    srcs_raw = rng.integers(0, n_accounts, n_transactions)
    dsts_raw = rng.integers(0, n_accounts, n_transactions)
    mask     = srcs_raw != dsts_raw
    srcs_raw, dsts_raw = srcs_raw[mask], dsts_raw[mask]

    pair_amounts: dict = {}
    for s, d in zip(srcs_raw.tolist(), dsts_raw.tolist()):
        # Fraud accounts send anomalously large/erratic amounts
        amt = float(rng.lognormal(11.5, 2.0) if s in fraud_accs
                    else rng.lognormal(9.0, 0.8))
        pair_amounts.setdefault((s, d), []).append(amt)

    out_count = {}
    for (s, d), amts in pair_amounts.items():
        out_count[s] = out_count.get(s, 0) + len(amts)

    edges_src, edges_dst, edge_feats_list, edge_labels_list = [], [], [], []
    fraud_edge_count = np.zeros(n_accounts, dtype=np.int64)
    total_edge_count = np.zeros(n_accounts, dtype=np.int64)

    for (i, j), amts in pair_amounts.items():
        F_ij     = float(amts[-1])
        T_avg_ij = float(np.mean(amts))
        sigma_ij = float(np.std(amts)) + 1e-6
        Delta_ij = abs(F_ij - T_avg_ij)
        rel_dev  = Delta_ij / (T_avg_ij + 1e-6)
        R_ij     = len(amts) / max(out_count.get(i, 1), 1)
        eps_ij   = 0.5*eps_n[i] + 0.5*eps_n[j] + 0.1*sigma_ij
        ci, cj   = int(cat_idx[i]), int(cat_idx[j])
        cat_mis  = 1.0 if ci != cj else 0.5
        AS_ij    = rel_dev * cat_mis / (eps_ij + 1e-6)
        T_i      = STATIC_THRESHOLDS[ACCOUNT_TYPES[ci]]

        if i in fraud_accs:
            p_fraud = max(float(torch.sigmoid(torch.tensor(5.0*(AS_ij - T_i*0.5)))), 0.75)
        else:
            p_fraud = min(float(torch.sigmoid(torch.tensor(5.0*(AS_ij - T_i*1.8)))), 0.05)

        is_fraud = int(rng.random() < p_fraud)

        feat = [
            H[i], H[j], age[i], age[j], loc[i], loc[j],
            float(np.clip(T_avg_n[i]/1e5, 0, 1)),
            float(np.clip(T_avg_n[j]/1e5, 0, 1)),
            float(np.clip(T_avg_ij/1e5, 0, 1)),
            eps_n[i], eps_n[j],
            float(np.clip(eps_ij/5.0, 0, 1)),
            ci/(NUM_TYPES-1), cj/(NUM_TYPES-1),
            float(np.clip(F_ij/1e6, 0, 1)),
            float(np.clip(R_ij, 0, 1)),
            float(np.clip(rel_dev, 0, 5)),
        ]

        edges_src.append(i); edges_dst.append(j)
        edge_feats_list.append(feat); edge_labels_list.append(is_fraud)
        total_edge_count[i] += 1
        if is_fraud:
            fraud_edge_count[i] += 1

    edge_feats_np = StandardScaler().fit_transform(
        np.array(edge_feats_list, dtype=np.float32))

    A  = np.zeros((n_accounts, n_accounts), dtype=np.float32)
    W1 = np.zeros((n_accounts, n_accounts), dtype=np.float32)
    for s, d in zip(edges_src, edges_dst):
        A[s, d]  = 1.0
        W1[s, d] = float(W0[int(cat_idx[s]), int(cat_idx[d])])

    # Node fraud label: fraud account AND >40% of its edges are fraud
    node_labels = np.zeros(n_accounts, dtype=np.int64)
    for i in range(n_accounts):
        if total_edge_count[i] > 0:
            ratio = fraud_edge_count[i] / total_edge_count[i]
            if ratio > 0.40:
                node_labels[i] = 1

    node_feats_np = np.stack(
        [H, age, loc, np.clip(T_avg_n/1e5,0,1), eps_n, cat_idx/(NUM_TYPES-1)],
        axis=1).astype(np.float32)

    fraud_pct = node_labels.mean() * 100
    print(f"Nodes: {n_accounts}")
    print(f"Edges: {len(edges_src)}")
    print(f"Fraud Nodes: {node_labels.sum()} ({fraud_pct:.2f}%)")

    return {
        "n_accounts":    n_accounts,
        "node_feats":    torch.tensor(node_feats_np,           device=device),
        "edge_feats":    torch.tensor(edge_feats_np,           device=device),
        "edges_src":     torch.tensor(edges_src,               device=device),
        "edges_dst":     torch.tensor(edges_dst,               device=device),
        "node_labels":   torch.tensor(node_labels,             device=device),
        "edge_labels":   torch.tensor(edge_labels_list,        device=device),
        "account_types": torch.tensor(cat_idx,                 device=device),
        "A":             torch.tensor(A,                       device=device),
        "W1":            torch.tensor(W1,                      device=device),
    }

# ============================================================
# SPARSE SOFTMAX  (version-safe)
# ============================================================
def sparse_softmax(scores, src_idx, N):
    try:
        max_s = torch.full((N,), -1e9, device=scores.device, dtype=scores.dtype)
        max_s.scatter_reduce_(0, src_idx, scores, reduce="amax", include_self=True)
    except Exception:
        max_s = torch.full((N,), -1e9, device=scores.device, dtype=scores.dtype)
        for e in range(scores.size(0)):
            n = int(src_idx[e])
            if float(scores[e]) > float(max_s[n]):
                max_s[n] = scores[e]
    exp_s = torch.exp(scores - max_s[src_idx].detach())
    sum_s = torch.zeros(N, device=scores.device, dtype=scores.dtype)
    sum_s.index_add_(0, src_idx, exp_s)
    return exp_s / (sum_s[src_idx] + 1e-12)

# ============================================================
# FFD-STRA-GAT LAYER
# ============================================================
class FFDSTRAGATLayer(nn.Module):
    """
    z̃_ij  = [h_i, h_j, e_ij, w_ij]   (paper §VII)
    Ats_ij = LeakyReLU(aᵀ z̃_ij)
    R_ij   = softmax_j(Ats_ij)
    α_ij   = σ(bᵀ z̃_ij),  Ψ_ij = softplus(dᵀ z̃_ij)
    ε_ij   = α_ij·ε_i + (1−α_ij)·ε_j + 0.1·Ψ_ij·σ_ij
    λ_ij   = softplus(cᵀ z̃_ij)
    ρ_i    = Σ_j R_ij · msg(h_j, e_ij)
    """
    def __init__(self, node_dim, edge_dim, hidden, dropout=0.30):
        super().__init__()
        self.hidden   = hidden
        self.node_proj = nn.Linear(node_dim, hidden)
        self.edge_proj = nn.Linear(edge_dim, hidden)
        dz             = hidden * 3 + 1          # [h_i, h_j, e_ij, w_ij]
        self.attn_fc   = nn.Linear(dz, 1, bias=False)
        self.alpha_fc  = nn.Linear(dz, 1, bias=False)
        self.psi_fc    = nn.Linear(dz, 1, bias=False)
        self.lambda_fc = nn.Linear(dz, 1, bias=False)
        self.msg_fc    = nn.Linear(hidden * 2, hidden)
        self.update_fc = nn.Linear(hidden * 2, hidden)
        self.norm      = nn.LayerNorm(hidden)
        self.dropout   = nn.Dropout(dropout)
        self.lrelu     = nn.LeakyReLU(0.2)

    def forward(self, h, edge_emb, edges_src, edges_dst,
                w_ij, sigma_ij, eps_i, eps_j):
        N      = h.size(0)
        h_proj = F.elu(self.node_proj(h))
        e_ij   = F.elu(self.edge_proj(edge_emb))
        h_i    = h_proj[edges_src]
        h_j    = h_proj[edges_dst]

        z = torch.cat([h_i, h_j, e_ij, w_ij.unsqueeze(1)], dim=1)

        att    = self.lrelu(self.attn_fc(z)).squeeze(-1)
        R_ij   = sparse_softmax(att, edges_src, N)

        alpha  = torch.sigmoid(self.alpha_fc(z)).squeeze(-1)
        psi    = F.softplus(self.psi_fc(z)).squeeze(-1)
        eps_ij = (alpha*eps_i + (1-alpha)*eps_j + 0.1*psi*sigma_ij).clamp(1e-5, 5.0)
        lam    = F.softplus(self.lambda_fc(z)).squeeze(-1).clamp(max=10.0)

        msg = self.msg_fc(torch.cat([h_j, e_ij], dim=1))
        rho = torch.zeros(N, self.hidden, device=h.device)
        rho.index_add_(0, edges_src, R_ij.unsqueeze(1) * msg)

        update = F.elu(self.update_fc(torch.cat([h_proj, rho], dim=1)))
        h_next = self.dropout(self.norm(h_proj + update))
        return h_next, R_ij, eps_ij, lam, att

# ============================================================
# FFD-STRA-GAT MODEL
# ============================================================
class FFDSTRAGATModel(nn.Module):
    def __init__(self, node_in=6, edge_in=17, hidden=64,
                 n_layers=3, dropout=0.30, type_emb_dim=16):
        super().__init__()
        self.hidden = hidden

        self.type_emb    = nn.Embedding(NUM_TYPES, type_emb_dim)
        self.node_encoder = nn.Sequential(
            nn.Linear(node_in + type_emb_dim, hidden*2), nn.ELU(),
            nn.Linear(hidden*2, hidden), nn.LayerNorm(hidden))
        self.edge_encoder = nn.Sequential(
            nn.Linear(edge_in, hidden*2), nn.ELU(),
            nn.Linear(hidden*2, hidden), nn.LayerNorm(hidden))
        self.layers = nn.ModuleList([
            FFDSTRAGATLayer(hidden, hidden, hidden, dropout)
            for _ in range(n_layers)])

        # Classifier branch
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.ELU(),
            nn.Linear(hidden//2, 1))

        # Anomaly branch: h + anomaly_score + log_degree + eps_i
        self.anomaly_mlp = nn.Sequential(
            nn.Linear(hidden+3, hidden), nn.ELU(),
            nn.Linear(hidden, hidden//2), nn.ELU(),
            nn.Linear(hidden//2, 1))

        # Learnable fusion weight
        self.fusion_w = nn.Parameter(torch.tensor(0.5))

    def forward(self, node_feats, edge_feats, edges_src, edges_dst,
                account_types, W1, A):
        N     = node_feats.size(0)
        t_emb = self.type_emb(account_types)
        h     = self.node_encoder(torch.cat([node_feats, t_emb], dim=1))
        e_emb = self.edge_encoder(edge_feats)

        w_ij     = W1[edges_src, edges_dst]
        sigma_ij = edge_feats[:, 16].abs() + 1e-6
        eps_i    = node_feats[edges_src, 4]
        eps_j    = node_feats[edges_dst, 4]

        last_eps = None
        for layer in self.layers:
            h, R_ij, last_eps, lam, _ = layer(
                h, e_emb, edges_src, edges_dst,
                w_ij, sigma_ij, eps_i, eps_j)

        clf_logit = self.classifier(h).squeeze(1)

        # Anomaly score per node  (AS^(W,t)_i via w_ij · σ_ij aggregation)
        anom = torch.zeros(N, device=h.device)
        anom.index_add_(0, edges_src, w_ij * sigma_ij)
        deg  = torch.zeros(N, device=h.device)
        deg.index_add_(0, edges_src, torch.ones(edges_src.size(0), device=h.device))
        anom = anom / deg.clamp(min=1)
        anom = anom / (anom.std() + 1e-6)

        anom_input  = torch.cat([
            h,
            anom.unsqueeze(1),
            torch.log1p(deg).unsqueeze(1),
            node_feats[:, 4].unsqueeze(1),
        ], dim=1)
        anom_logit  = self.anomaly_mlp(anom_input).squeeze(1)

        alpha       = torch.sigmoid(self.fusion_w)
        final_logit = alpha * clf_logit + (1 - alpha) * anom_logit
        return torch.sigmoid(final_logit), anom, deg

# ============================================================
# GCN BASELINE
# ============================================================
class GCNLayer(nn.Module):
    def __init__(self, in_d, out_d):
        super().__init__()
        self.lin  = nn.Linear(in_d, out_d, bias=False)
        self.norm = nn.BatchNorm1d(out_d)
    def forward(self, h, A_norm):
        return F.relu(self.norm(A_norm @ self.lin(h)))

class GCNModel(nn.Module):
    def __init__(self, in_d=6, hidden=64, n_layers=3, dropout=0.30):
        super().__init__()
        dims        = [in_d] + [hidden]*n_layers
        self.layers = nn.ModuleList([GCNLayer(dims[i], dims[i+1]) for i in range(n_layers)])
        self.cls    = nn.Linear(hidden, 1)
        self.drop   = nn.Dropout(dropout)

    def _norm_adj(self, A):
        N     = A.size(0)
        A_hat = A + torch.eye(N, device=A.device)
        D     = A_hat.sum(1).clamp(1); Di = D.pow(-0.5)
        return Di.unsqueeze(1) * A_hat * Di.unsqueeze(0)

    def forward(self, node_feats, A):
        A_norm = self._norm_adj(A)
        h = node_feats
        for layer in self.layers:
            h = self.drop(layer(h, A_norm))
        return torch.sigmoid(self.cls(h).squeeze(1))

# ============================================================
# GRAPHSAGE BASELINE
# ============================================================
class SAGELayer(nn.Module):
    def __init__(self, in_d, out_d):
        super().__init__()
        self.lin  = nn.Linear(in_d*2, out_d)
        self.norm = nn.BatchNorm1d(out_d)
    def forward(self, h, edges_src, edges_dst, N):
        agg   = torch.zeros(N, h.size(1), device=h.device)
        count = torch.zeros(N, 1,         device=h.device)
        agg.index_add_(0, edges_src, h[edges_dst])
        count.index_add_(0, edges_src, torch.ones(edges_src.size(0), 1, device=h.device))
        return F.relu(self.norm(self.lin(torch.cat([h, agg/count.clamp(1)], dim=1))))

class GraphSAGEModel(nn.Module):
    def __init__(self, in_d=6, hidden=64, n_layers=3, dropout=0.30):
        super().__init__()
        dims        = [in_d] + [hidden]*n_layers
        self.layers = nn.ModuleList([SAGELayer(dims[i], dims[i+1]) for i in range(n_layers)])
        self.cls    = nn.Linear(hidden, 1)
        self.drop   = nn.Dropout(dropout)
    def forward(self, node_feats, edges_src, edges_dst, N):
        h = node_feats
        for layer in self.layers:
            h = self.drop(layer(h, edges_src, edges_dst, N))
        return torch.sigmoid(self.cls(h).squeeze(1))

# ============================================================
# FOCAL LOSS  (with label smoothing + dynamic alpha)
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, smoothing=0.02):
        super().__init__()
        self.gamma     = gamma
        self.smoothing = smoothing

    def forward(self, pred, target):
        pred   = pred.clamp(1e-7, 1-1e-7)
        target = target*(1-self.smoothing) + 0.5*self.smoothing
        pos    = target.sum(); neg = len(target) - pos
        alpha  = neg / (pos + neg + 1e-6)
        pt     = torch.where(target > 0.5, pred, 1-pred)
        w      = torch.where(target > 0.5, alpha, 1-alpha)
        return (-w * (1-pt).pow(self.gamma) * torch.log(pt)).mean()

# ============================================================
# EARLY STOPPING
# ============================================================
class EarlyStopping:
    def __init__(self, patience=40, min_delta=1e-4):
        self.patience  = patience
        self.min_delta = min_delta
        self.best      = None
        self.counter   = 0
        self.stop      = False

    def __call__(self, score):
        if self.best is None or score > self.best + self.min_delta:
            self.best = score; self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true, y_prob, threshold=0.50):
    y_pred   = (y_prob >= threshold).astype(int)
    has_both = len(np.unique(y_true)) > 1
    return {
        "AUC-ROC":          roc_auc_score(y_true, y_prob)           if has_both else 0.5,
        "AP":               average_precision_score(y_true, y_prob)  if has_both else 0.0,
        "F1":               f1_score(y_true, y_pred,       zero_division=0),
        "Precision":        precision_score(y_true, y_pred, zero_division=0),
        "Recall":           recall_score(y_true, y_pred,   zero_division=0),
        "Balanced Accuracy":balanced_accuracy_score(y_true, y_pred),
        "CM":               confusion_matrix(y_true, y_pred, labels=[0,1]),
    }

def find_best_threshold(y_true, y_prob):
    best_thr, best_f1 = 0.50, -1
    for thr in np.arange(0.05, 0.96, 0.01):
        f1 = f1_score(y_true, (y_prob >= thr).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return best_thr, best_f1

# ============================================================
# STRATIFIED TRAIN / VAL / TEST SPLIT
# ============================================================
def make_splits(labels_np, train_ratio=0.60, val_ratio=0.20, seed=42):
    """
    Stratified 60/20/20 split.
    FIX: previously create_split() was NOT stratified, causing
    val folds with very few fraud examples → trivially AUC=1.0 → early stop.
    """
    idx = np.arange(len(labels_np))
    train_idx, temp_idx = train_test_split(
        idx, test_size=1-train_ratio, stratify=labels_np, random_state=seed)
    relative_val = val_ratio / (1 - train_ratio)
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=1-relative_val,
        stratify=labels_np[temp_idx], random_state=seed)
    return train_idx, val_idx, test_idx

# ============================================================
# POS-WEIGHT HELPER
# ============================================================
def pos_weight(labels):
    n1 = (labels==1).sum().float().clamp(1)
    n0 = (labels==0).sum().float()
    return n0 / n1

# ============================================================
# TRAIN FFD-STRA-GAT
# ============================================================
def train_ffd(data, cfg):
    dev     = data["node_feats"].device
    labels  = data["node_labels"].float()
    labels_np = labels.cpu().numpy()

    train_idx, val_idx, test_idx = make_splits(
        labels_np, cfg["train_ratio"], cfg["val_ratio"])
    train_t = torch.tensor(train_idx, device=dev)
    val_t   = torch.tensor(val_idx,   device=dev)

    model = FFDSTRAGATModel(
        node_in  = data["node_feats"].shape[1],
        edge_in  = data["edge_feats"].shape[1],
        hidden   = cfg["hidden_dim"],
        n_layers = cfg["num_layers"],
        dropout  = cfg["dropout"],
    ).to(dev)

    opt   = optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=10)
    crit  = FocalLoss(gamma=2.0, smoothing=0.02)
    stopper = EarlyStopping(patience=cfg["patience"])

    best_auc, best_state, history = -1, None, []
    t0 = time.time()

    for epoch in range(1, cfg["epochs"]+1):
        model.train(); opt.zero_grad()
        probs, _, _ = model(
            data["node_feats"], data["edge_feats"],
            data["edges_src"],  data["edges_dst"],
            data["account_types"], data["W1"], data["A"])
        loss = crit(probs[train_t], labels[train_t])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        model.eval()
        with torch.no_grad():
            vp, _, _ = model(
                data["node_feats"], data["edge_feats"],
                data["edges_src"],  data["edges_dst"],
                data["account_types"], data["W1"], data["A"])
        vp_np  = vp[val_t].cpu().numpy()
        vt_np  = labels[val_t].cpu().numpy()
        val_auc = roc_auc_score(vt_np, vp_np) if len(np.unique(vt_np)) > 1 else 0.5
        sched.step(val_auc)
        history.append(loss.item())

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        stopper(val_auc)
        if epoch % 20 == 0:
            _, bf1 = find_best_threshold(vt_np, vp_np)
            print(f"[FFD] Epoch {epoch:3d} | Loss {loss.item():.4f} "
                  f"| ValAUC {val_auc:.4f} | BestF1 {bf1:.4f}")
        if stopper.stop:
            print(f"Early stopping at epoch {epoch}"); break

    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        final, _, _ = model(
            data["node_feats"], data["edge_feats"],
            data["edges_src"],  data["edges_dst"],
            data["account_types"], data["W1"], data["A"])
    probs_np = final.cpu().numpy()
    best_thr, _ = find_best_threshold(labels_np, probs_np)
    metrics = compute_metrics(labels_np, probs_np, best_thr)
    print(f"\nBest threshold = {best_thr:.3f}")
    print(f"Finished in {time.time()-t0:.2f}s")
    return model, metrics, history, probs_np, best_thr

# ============================================================
# TRAIN GCN
# ============================================================
def train_gcn(data, cfg):
    dev      = data["node_feats"].device
    labels   = data["node_labels"].float()
    labels_np= labels.cpu().numpy()

    train_idx, val_idx, test_idx = make_splits(
        labels_np, cfg["train_ratio"], cfg["val_ratio"])
    train_t = torch.tensor(train_idx, device=dev)
    val_t   = torch.tensor(val_idx,   device=dev)

    model = GCNModel(
        in_d=data["node_feats"].shape[1],
        hidden=cfg["hidden_dim"], n_layers=cfg["num_layers"],
        dropout=cfg["dropout"]).to(dev)

    opt   = optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=10)
    pw    = pos_weight(labels[train_t])   # FIX: GCN now uses pos_weight
    stopper = EarlyStopping(patience=cfg["patience"])
    best_auc, best_state, history = -1, None, []
    t0 = time.time()

    for epoch in range(1, cfg["epochs"]+1):
        model.train(); opt.zero_grad()
        probs = model(data["node_feats"], data["A"])
        # Weighted BCE — upweights the minority fraud class
        loss  = F.binary_cross_entropy(
            probs[train_t], labels[train_t],
            weight=torch.where(
                labels[train_t]==1,
                pw.to(dev), torch.ones(1, device=dev)))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        model.eval()
        with torch.no_grad():
            vp = model(data["node_feats"], data["A"])
        vp_np  = vp[val_t].cpu().numpy()
        vt_np  = labels[val_t].cpu().numpy()
        val_auc = roc_auc_score(vt_np, vp_np) if len(np.unique(vt_np)) > 1 else 0.5
        sched.step(val_auc)
        history.append(loss.item())

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        stopper(val_auc)
        if epoch % 20 == 0:
            print(f"[GCN] Epoch {epoch:3d} | Loss {loss.item():.4f} | ValAUC {val_auc:.4f}")
        if stopper.stop:
            print(f"GCN Early stopping at epoch {epoch}"); break

    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        probs = model(data["node_feats"], data["A"])
    probs_np = probs.cpu().numpy()
    best_thr, _ = find_best_threshold(labels_np, probs_np)
    metrics = compute_metrics(labels_np, probs_np, best_thr)
    print(f"Finished in {time.time()-t0:.2f}s")
    return model, metrics, history, probs_np, best_thr

# ============================================================
# TRAIN GRAPHSAGE
# ============================================================
def train_sage(data, cfg):
    dev      = data["node_feats"].device
    N        = data["n_accounts"]
    labels   = data["node_labels"].float()
    labels_np= labels.cpu().numpy()

    train_idx, val_idx, test_idx = make_splits(
        labels_np, cfg["train_ratio"], cfg["val_ratio"])
    train_t = torch.tensor(train_idx, device=dev)
    val_t   = torch.tensor(val_idx,   device=dev)

    model = GraphSAGEModel(
        in_d=data["node_feats"].shape[1],
        hidden=cfg["hidden_dim"], n_layers=cfg["num_layers"],
        dropout=cfg["dropout"]).to(dev)

    opt   = optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=10)
    pw    = pos_weight(labels[train_t])
    stopper = EarlyStopping(patience=cfg["patience"])
    best_auc, best_state, history = -1, None, []
    t0 = time.time()

    for epoch in range(1, cfg["epochs"]+1):
        model.train(); opt.zero_grad()
        probs = model(data["node_feats"], data["edges_src"], data["edges_dst"], N)
        loss  = F.binary_cross_entropy(
            probs[train_t], labels[train_t],
            weight=torch.where(
                labels[train_t]==1,
                pw.to(dev), torch.ones(1, device=dev)))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        model.eval()
        with torch.no_grad():
            vp = model(data["node_feats"], data["edges_src"], data["edges_dst"], N)
        vp_np  = vp[val_t].cpu().numpy()
        vt_np  = labels[val_t].cpu().numpy()
        val_auc = roc_auc_score(vt_np, vp_np) if len(np.unique(vt_np)) > 1 else 0.5
        sched.step(val_auc)
        history.append(loss.item())

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        stopper(val_auc)
        if epoch % 20 == 0:
            print(f"[GraphSAGE] Epoch {epoch:3d} | Loss {loss.item():.4f} | ValAUC {val_auc:.4f}")
        if stopper.stop:
            print(f"GraphSAGE Early stopping at epoch {epoch}"); break

    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        probs = model(data["node_feats"], data["edges_src"], data["edges_dst"], N)
    probs_np = probs.cpu().numpy()
    best_thr, _ = find_best_threshold(labels_np, probs_np)
    metrics = compute_metrics(labels_np, probs_np, best_thr)
    print(f"Finished in {time.time()-t0:.2f}s")
    return model, metrics, history, probs_np, best_thr

# ============================================================
# DDP
# ============================================================
def _ddp_worker(rank, world_size, data_cpu, result_ns, cfg):
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "29501"
    dist.init_process_group("gloo", rank=rank, world_size=world_size)
    dev  = torch.device("cpu")
    data = {k: v.to(dev) if isinstance(v, torch.Tensor) else v for k,v in data_cpu.items()}

    model = FFDSTRAGATModel(
        node_in=data["node_feats"].shape[1],
        edge_in=data["edge_feats"].shape[1],
        hidden=cfg["hidden_dim"], n_layers=cfg["num_layers"],
        dropout=cfg["dropout"]).to(dev)
    ddp   = DDP(model)
    opt   = optim.AdamW(ddp.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=80, eta_min=1e-5)
    crit  = FocalLoss(gamma=2.0, smoothing=0.02)
    labels= data["node_labels"].float()
    N     = data["n_accounts"]
    shard = math.ceil(N / world_size)
    s, e  = rank*shard, min((rank+1)*shard, N)

    for ep in range(1, 81):
        ddp.train(); opt.zero_grad()
        probs, _, _ = ddp(data["node_feats"], data["edge_feats"],
                          data["edges_src"], data["edges_dst"],
                          data["account_types"], data["W1"], data["A"])
        loss = crit(probs[s:e], labels[s:e])
        loss.backward()
        nn.utils.clip_grad_norm_(ddp.parameters(), 1.0)
        opt.step(); sched.step()
        if rank==0 and ep%20==0:
            print(f"[DDP r0] ep {ep:3d} | loss {loss.item():.4f}")

    if rank == 0:
        ddp.eval()
        with torch.no_grad():
            p, _, _ = ddp(data["node_feats"], data["edge_feats"],
                          data["edges_src"], data["edges_dst"],
                          data["account_types"], data["W1"], data["A"])
        thr, _ = find_best_threshold(labels.numpy(), p.numpy())
        result_ns["ddp_metrics"] = compute_metrics(labels.numpy(), p.numpy(), thr)
    dist.destroy_process_group()

def run_ddp(data_cpu, cfg):
    mgr = mp.Manager(); res = mgr.dict()
    ctx = mp.get_context("spawn")
    ws  = cfg["world_size"]
    procs = [ctx.Process(target=_ddp_worker, args=(r, ws, data_cpu, res, cfg))
             for r in range(ws)]
    for p in procs: p.start()
    for p in procs:
        p.join(timeout=300)
        if p.exitcode not in (0, None):
            print(f"  [DDP] worker {p.pid} exited {p.exitcode}")
    return dict(res)

# ============================================================
# PLOTS
# ============================================================
MODEL_COLORS = {"FFD-STRA-GAT":"#2ecc71", "GCN":"#e74c3c", "GraphSAGE":"#3498db"}
plt.rcParams.update({"font.size":11,"axes.titlesize":13,"axes.labelsize":11,"legend.fontsize":10})

def save_fig(fname):
    path = os.path.join(OUTPUT_DIR, fname)
    plt.savefig(path, dpi=300, bbox_inches="tight"); plt.close()
    print(f"Saved → {path}")

def plot_loss_curves(histories, labels):
    plt.figure(figsize=(8,5))
    for h, lbl in zip(histories, labels):
        plt.plot(h, label=lbl, lw=2, color=MODEL_COLORS.get(lbl,"#888"))
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training Loss Curves")
    plt.legend(); plt.grid(alpha=0.3); save_fig("loss_curves.png")

def plot_metric_comparison(metrics_list, labels):
    keys = ["AUC-ROC","AP","F1","Precision","Recall","Balanced Accuracy"]
    fig, axes = plt.subplots(2, 3, figsize=(15,10))
    for ax, mk in zip(axes.flatten(), keys):
        vals = [m[mk] for m in metrics_list]
        cols = [MODEL_COLORS.get(l,"#888") for l in labels]
        bars = ax.bar(labels, vals, color=cols, alpha=0.85, edgecolor="k", lw=0.8)
        ax.set_ylim(0, 1.12); ax.set_title(mk); ax.grid(axis="y", alpha=0.3)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.02, f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")
    plt.tight_layout(); save_fig("metric_comparison.png")

def plot_confusion_matrices(metrics_list, labels):
    n = len(labels)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
    if n == 1: axes = [axes]
    for ax, m, lbl in zip(axes, metrics_list, labels):
        cm = m["CM"]; cm_n = cm.astype(float)/np.maximum(cm.sum(1,keepdims=True),1)
        ax.imshow(cm_n, cmap="Blues")
        ax.set_title(lbl, fontweight="bold")
        ax.set_xticks([0,1]); ax.set_yticks([0,1])
        ax.set_xticklabels(["Normal","Fraud"]); ax.set_yticklabels(["Normal","Fraud"])
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{cm[i,j]}\n({cm_n[i,j]:.2f})",
                        ha="center", va="center", fontsize=10)
    plt.tight_layout(); save_fig("confusion_matrices.png")

def plot_score_distributions(all_probs, true_labels, labels):
    n = len(labels)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
    if n == 1: axes = [axes]
    for ax, probs, lbl in zip(axes, all_probs, labels):
        ax.hist(probs[true_labels==0], bins=40, alpha=0.6, label="Normal", color="#3498db", density=True)
        ax.hist(probs[true_labels==1], bins=40, alpha=0.6, label="Fraud",  color="#e74c3c", density=True)
        ax.axvline(0.5, ls="--", color="k", lw=1.2)
        ax.set_title(lbl, fontweight="bold"); ax.set_xlabel("Probability")
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); save_fig("score_distributions.png")

def plot_roc_curves(all_probs, true_labels, labels, all_metrics):
    plt.figure(figsize=(7,6))
    plt.plot([0,1],[0,1],"k--", lw=1, label="Random")
    for probs, lbl, m in zip(all_probs, labels, all_metrics):
        fpr, tpr, _ = roc_curve(true_labels, probs)
        plt.plot(fpr, tpr, lw=2, label=f"{lbl} (AUC={m['AUC-ROC']:.3f})",
                 color=MODEL_COLORS.get(lbl,"#888"))
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC Curves")
    plt.legend(); plt.grid(alpha=0.3); save_fig("roc_curves.png")

def plot_pr_curves(all_probs, true_labels, labels, all_metrics):
    plt.figure(figsize=(7,6))
    for probs, lbl, m in zip(all_probs, labels, all_metrics):
        prec, rec, _ = precision_recall_curve(true_labels, probs)
        plt.plot(rec, prec, lw=2, label=f"{lbl} (AP={m['AP']:.3f})",
                 color=MODEL_COLORS.get(lbl,"#888"))
    plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall Curves")
    plt.legend(); plt.grid(alpha=0.3); save_fig("pr_curves.png")

# ============================================================
# SUMMARY TABLE
# ============================================================
def print_summary(all_metrics, labels, all_thresholds):
    w = 90
    print("\n" + "="*w)
    print(f"{'Model':<20} {'AUC':>8} {'AP':>8} {'F1':>8} {'Prec':>8} {'Recall':>8} {'BalAcc':>8}")
    print("="*w)
    for m, lbl in zip(all_metrics, labels):
        print(f"{lbl:<20} {m['AUC-ROC']:>8.4f} {m['AP']:>8.4f} {m['F1']:>8.4f} "
              f"{m['Precision']:>8.4f} {m['Recall']:>8.4f} {m['Balanced Accuracy']:>8.4f}")
    print("="*w)

    print("\nOptimal Thresholds")
    for lbl, thr in zip(labels, all_thresholds):
        print(f"  {lbl:<20}: {thr:.3f}")

    print("\nBest per metric:")
    for mk in ["AUC-ROC","AP","F1","Precision","Recall","Balanced Accuracy"]:
        bv, bl = max(((m[mk],l) for m,l in zip(all_metrics,labels)), key=lambda x:x[0])
        star = "★" if bl=="FFD-STRA-GAT" else " "
        print(f"  {star} {mk:<20}: {bl} ({bv:.4f})")

# ============================================================
# MAIN
# ============================================================
def main():
    print("="*80)
    print("FFD-STRA-GAT")
    print("Financial Fraud Detection using")
    print("Spatiotemporal Risk-Aware Graph Attention Networks")
    print("="*80)
    print(f"\nDevice : {DEVICE}")
    print(f"Epochs : {CFG['epochs']}")
    print(f"Hidden : {CFG['hidden_dim']}")
    print(f"Layers : {CFG['num_layers']}")
    print(f"Split  : {int(CFG['train_ratio']*100)}/{int(CFG['val_ratio']*100)}"
          f"/{int((1-CFG['train_ratio']-CFG['val_ratio'])*100)} (stratified train/val/test)")

    print("\nGenerating transaction graph...")
    data = generate_transaction_graph(
        n_accounts    = CFG["n_accounts"],
        n_transactions= CFG["n_transactions"],
        fraud_ratio   = CFG["fraud_ratio"],
        seed=42, device=DEVICE)

    all_metrics, all_hists, all_labels, all_probs, all_thresholds = [], [], [], [], []
    true_labels = data["node_labels"].cpu().numpy()

    # ── FFD-STRA-GAT ─────────────────────────────────────────
    print("\n" + "="*60)
    print("Training FFD-STRA-GAT")
    print("="*60)
    ffd_model, m_ffd, h_ffd, p_ffd, thr_ffd = train_ffd(data, CFG)
    all_metrics.append(m_ffd); all_hists.append(h_ffd)
    all_labels.append("FFD-STRA-GAT"); all_probs.append(p_ffd)
    all_thresholds.append(thr_ffd)

    # ── GCN ──────────────────────────────────────────────────
    print("\n" + "="*60)
    print("Training GCN")
    print("="*60)
    gcn_model, m_gcn, h_gcn, p_gcn, thr_gcn = train_gcn(data, CFG)
    all_metrics.append(m_gcn); all_hists.append(h_gcn)
    all_labels.append("GCN"); all_probs.append(p_gcn)
    all_thresholds.append(thr_gcn)

    # ── GraphSAGE ─────────────────────────────────────────────
    print("\n" + "="*60)
    print("Training GraphSAGE")
    print("="*60)
    sage_model, m_sage, h_sage, p_sage, thr_sage = train_sage(data, CFG)
    all_metrics.append(m_sage); all_hists.append(h_sage)
    all_labels.append("GraphSAGE"); all_probs.append(p_sage)
    all_thresholds.append(thr_sage)

    # ── DDP ──────────────────────────────────────────────────
    if CFG["use_ddp"]:
        print("\n" + "="*60)
        print(f"DDP Training ({CFG['world_size']} workers)")
        print("="*60)
        data_cpu = {k: v.cpu() if isinstance(v, torch.Tensor) else v for k,v in data.items()}
        try:
            t0  = time.time()
            res = run_ddp(data_cpu, CFG)
            dm  = res.get("ddp_metrics")
            print(f"DDP done in {time.time()-t0:.1f}s")
            if dm:
                print(f"DDP AUC-ROC {dm['AUC-ROC']:.4f} | F1 {dm['F1']:.4f}")
            else:
                print("DDP: no results collected.")
        except Exception as ex:
            print(f"DDP skipped: {ex}")

    # ── Summary ───────────────────────────────────────────────
    print_summary(all_metrics, all_labels, all_thresholds)

    def nparams(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
    print("\nParameter Counts")
    print(f"  FFD-STRA-GAT : {nparams(ffd_model):,}")
    print(f"  GCN          : {nparams(gcn_model):,}")
    print(f"  GraphSAGE    : {nparams(sage_model):,}")

    # ── Plots ─────────────────────────────────────────────────
    print("\nGenerating figures...")
    plot_loss_curves(all_hists, all_labels)
    plot_metric_comparison(all_metrics, all_labels)
    plot_confusion_matrices(all_metrics, all_labels)
    plot_score_distributions(all_probs, true_labels, all_labels)
    plot_roc_curves(all_probs, true_labels, all_labels, all_metrics)
    plot_pr_curves(all_probs, true_labels, all_labels, all_metrics)

    print(f"\nDone.\n\nOutputs saved to:\n{OUTPUT_DIR}")
    print("="*80)

if __name__ == "__main__":
    main()

FFD-STRA-GAT
Financial Fraud Detection using
Spatiotemporal Risk-Aware Graph Attention Networks

Device : cuda
Epochs : 250
Hidden : 64
Layers : 3
Split  : 60/20/20 (stratified train/val/test)

Generating transaction graph...
Nodes: 800
Edges: 5957
Fraud Nodes: 111 (13.88%)

Training FFD-STRA-GAT
[FFD] Epoch  20 | Loss 0.0244 | ValAUC 0.9611 | BestF1 0.8421
[FFD] Epoch  40 | Loss 0.0036 | ValAUC 0.9993 | BestF1 0.9778
[FFD] Epoch  60 | Loss 0.0033 | ValAUC 0.9993 | BestF1 0.9778
Early stopping at epoch 73

Best threshold = 0.740
Finished in 3.58s

Training GCN
[GCN] Epoch  20 | Loss 1.1595 | ValAUC 0.5847
[GCN] Epoch  40 | Loss 1.1572 | ValAUC 0.5731
[GCN] Epoch  60 | Loss 1.1779 | ValAUC 0.4651
GCN Early stopping at epoch 73
Finished in 0.70s

Training GraphSAGE
[GraphSAGE] Epoch  20 | Loss 1.2100 | ValAUC 0.5431
[GraphSAGE] Epoch  40 | Loss 1.1449 | ValAUC 0.5619
[GraphSAGE] Epoch  60 | Loss 1.1285 | ValAUC 0.5428
GraphSAGE Early stopping at epoch 78
Finished in 1.06s

DDP Training (